# 01 · SQL tools for an agent

You give an agent three tools over the database of a small online store (`data/shop.db`, built by
`data/seed_shop.py`) and watch how it **chooses** a tool, **calls** it and **uses the result**.

| Tool | What it does |
|---|---|
| `list_tables()` | names of the tables |
| `describe_table(table)` | columns and 3 sample rows |
| `run_query(sql)` | one read-only `SELECT`, at most 50 rows |

`create_agent` runs the loop of Session 02: the model node returns tool calls, the tools node runs them and appends one
`ToolMessage` per call, until the model answers without a tool call.

<img src="images/b3_agent_loop.png" width="640" alt="create_agent loop">

Docs: [Tools](https://docs.langchain.com/oss/python/langchain/tools) ·
[Agents](https://docs.langchain.com/oss/python/langchain/agents) ·
[SQL agent](https://docs.langchain.com/oss/python/langchain/sql-agent)

In [1]:
import json
import logging
import os
import re
import sqlite3
import time
from contextlib import closing
from pathlib import Path

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
for noisy in ("httpx", "httpx2", "mcp", "fastmcp"):  # HTTP and MCP clients log every request at INFO
    logging.getLogger(noisy).setLevel(logging.WARNING)
logger = logging.getLogger(__name__)

# ── Keys and model (.env of this session folder) ─────────────────────────────
load_dotenv()
for name in ("OPENAI_API_KEY",):
    if not os.getenv(name, "").strip():
        raise RuntimeError(f"Missing {name}: copy .env.example to .env in the session folder and fill it in")
model = init_chat_model(os.getenv("MODEL", "openai:gpt-6-luna"))
logger.info("Model: %s", model.model_name)

DB = Path("../data/shop.db")  # 200 customers, 40 products, 1,500 orders (Jan–Sep 2026), prices in VND
if not DB.exists():  # sqlite3.connect would create an empty file at a wrong path
    raise RuntimeError(f"No database at {DB}: open this notebook from the session folder, or run uv run data/seed_shop.py")
logger.info("Database: %s", DB)

INFO | Model: gpt-6-luna
INFO | Database: ../data/shop.db


## 1. Tools
A tool is a typed, documented Python function; `@tool` turns it into a **tool definition** (name, description,
JSON Schema of the arguments). Read-only is enforced in code, in four layers, never only in the prompt:

1. the query must start with `SELECT` or `WITH`;
2. the file is opened read-only (`mode=ro`, `PRAGMA query_only`);
3. a sqlite3 **authorizer** allows only reads (no `INSERT`, `UPDATE`, `DELETE`, `ATTACH`, `PRAGMA`);
4. at most 50 rows and a 2-second budget.

<img src="images/b3_sql_layers.png" width="640" alt="four read-only layers">

Errors are raised as `ToolException` with a message written for the model: what is wrong and how to fix the call.
Each tool logs one line per call with the normal `logger`: `run_query('SELECT …') -> ok 3 ms`.

In [2]:
from langchain.tools import tool
from langchain_core.tools import ToolException

MAX_ROWS, BUDGET_SECONDS = 50, 2.0


def log_call(tool: str, args: str, status: str, started: float) -> None:
    """The one log line of a tool call: tool(short args) -> ok|error <ms> ms."""
    args = " ".join(args.split())
    logger.info("%s(%s) -> %s %d ms", tool, args if len(args) <= 70 else args[:70] + "…", status,
                (time.monotonic() - started) * 1000)


def connect_readonly() -> sqlite3.Connection:
    con = sqlite3.connect(DB.resolve().as_uri() + "?mode=ro", uri=True)  # layer 2: read-only file
    con.execute("PRAGMA query_only = ON")                                  # layer 2: read-only session
    con.row_factory = sqlite3.Row
    return con  # close it with closing(...): `with con:` only ends a transaction


def table_names(con) -> list[str]:
    return [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name")]


def read_only(action, arg1, arg2, db, source):  # layer 3: sqlite3 authorizer
    allowed = (sqlite3.SQLITE_SELECT, sqlite3.SQLITE_READ, sqlite3.SQLITE_FUNCTION, sqlite3.SQLITE_RECURSIVE)
    return sqlite3.SQLITE_OK if action in allowed else sqlite3.SQLITE_DENY


@tool
def list_tables() -> list[str]:
    """List the tables of the shop database. Call it first, then describe_table."""
    started, status = time.monotonic(), "error"
    try:
        with closing(connect_readonly()) as con:
            tables = table_names(con)
        status = "ok"
        return tables
    finally:
        log_call("list_tables", "", status, started)


@tool
def describe_table(table: str) -> dict:
    """Show the columns of one table and 3 sample rows. Call it before writing SQL on that table."""
    started, status = time.monotonic(), "error"
    try:
        with closing(connect_readonly()) as con:
            if table not in table_names(con):
                raise ToolException(f"Unknown table '{table}'. Tables: {', '.join(table_names(con))}.")
            columns = [{"name": r["name"], "type": r["type"]} for r in con.execute(f"PRAGMA table_info({table})")]
            sample = [dict(r) for r in con.execute(f"SELECT * FROM {table} LIMIT 3")]
        status = "ok"
        return {"table": table, "columns": columns, "sample_rows": sample}
    finally:
        log_call("describe_table", repr(table), status, started)


def select(query: str) -> dict:
    if not re.match(r"(?is)^(select|with)\b", query):  # layer 1
        raise ToolException("Only SELECT queries are allowed (read-only). Rewrite it as SELECT ... FROM ...")
    with closing(connect_readonly()) as con:
        schema = "; ".join(f"{t}({', '.join(r['name'] for r in con.execute(f'PRAGMA table_info({t})'))})"
                           for t in table_names(con))
        con.set_authorizer(read_only)
        deadline = time.monotonic() + BUDGET_SECONDS
        con.set_progress_handler(lambda: time.monotonic() > deadline, 10_000)  # layer 4: time budget
        try:
            # layer 4: row cap. The newline before ")" keeps a trailing "-- comment" from swallowing the LIMIT.
            rows = [dict(r) for r in con.execute(f"SELECT * FROM ({query}\n) LIMIT {MAX_ROWS + 1}")]
        except sqlite3.Error as e:
            if "no such column" in str(e) or "no such table" in str(e):
                raise ToolException(f"{e}. Valid columns: {schema}. Call describe_table to see sample rows.") from e
            if "interrupted" in str(e):
                raise ToolException(f"The query took longer than {BUDGET_SECONDS:.0f} s. Add a WHERE clause or aggregate.") from e
            if "not authorized" in str(e):
                raise ToolException("Only reading is allowed (read-only). Rewrite it as a single SELECT.") from e
            raise ToolException(f"SQL error: {e}. Check the query against describe_table.") from e
    result = {"rows": rows[:MAX_ROWS], "row_count": min(len(rows), MAX_ROWS)}
    if len(rows) > MAX_ROWS:
        result["note"] = f"Only the first {MAX_ROWS} rows are shown. Aggregate (GROUP BY) or add a WHERE clause."
    return result


@tool
def run_query(sql: str) -> dict:
    """Run one read-only SQLite SELECT on the shop database and return at most 50 rows.
    Prices are in VND; orders.created_at is an ISO timestamp. Aggregate with GROUP BY instead of reading many rows."""
    started, status = time.monotonic(), "error"
    query = sql.strip().rstrip(";")
    try:
        result = select(query)
        status = "ok"
        return result
    finally:
        log_call("run_query", repr(query), status, started)


TOOLS = [list_tables, describe_table, run_query]


### What the model receives
`create_agent` sends this definition with every request (OpenAI "function" format). The model never sees the Python body.

In [3]:
from langchain_core.utils.function_calling import convert_to_openai_tool

print(json.dumps(convert_to_openai_tool(describe_table), indent=2))

{
  "type": "function",
  "function": {
    "name": "describe_table",
    "description": "Show the columns of one table and 3 sample rows. Call it before writing SQL on that table.",
    "parameters": {
      "properties": {
        "table": {
          "type": "string"
        }
      },
      "required": [
        "table"
      ],
      "type": "object"
    }
  }
}


## 2. Agent
`ToolErrorMiddleware` returns our `ToolException` messages to the model as `ToolMessage(status="error")`, so the model
can fix its call; other exceptions still stop the run.
Docs: [built-in middleware](https://docs.langchain.com/oss/python/langchain/middleware/built-in)

**Logging vs streaming.** *Logging* is diagnostics for the developer: lines your code and the servers write with
`logger.info` (`INFO | …`), kept or dropped by level. *Streaming* is the agent's own output delivered to the user while
it is produced: `stream()` uses `agent.astream(..., stream_mode=["updates", "messages"])`, prints one line per tool
call (`→`) and tool result (`←`) from `updates`, and the answer token by token from `messages`, instead of waiting
silently for the whole run. Docs: [Streaming](https://docs.langchain.com/oss/python/langchain/streaming)

In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolErrorMiddleware
from langchain.messages import AIMessageChunk, ToolMessage


def on_error(exc: Exception, request) -> str | None:
    if isinstance(exc, ToolException):
        return str(exc)  # the message written for the model; anything else propagates


agent = create_agent(
    model,
    tools=TOOLS,
    system_prompt=("You are a data analyst for an online store. Answer from the shop database only: look at the "
                   "tables you need, then write one SQL query per question when possible. Amounts are in VND; "
                   "exclude cancelled orders from revenue. Answer briefly, with the numbers."),
    middleware=[ToolErrorMiddleware(on_error)],
)


def short(text: str, n: int = 110) -> str:
    return text if len(text) <= n else text[:n] + "…"


async def stream(agent, question: str) -> list:
    """Stream one run: a line per tool call and tool result ("updates"), the answer token by token ("messages").
    Returns the messages the run added."""
    added, mid_line = [], False
    async for chunk in agent.astream({"messages": [{"role": "user", "content": question}]},
                                     stream_mode=["updates", "messages"], version="v2"):
        if chunk["type"] == "messages":  # (token, metadata): print the model's text as it is generated
            token, meta = chunk["data"]
            if isinstance(token, AIMessageChunk) and meta["langgraph_node"] == "model" and token.text:
                print(token.text, end="", flush=True)
                mid_line = True
            continue
        for update in chunk["data"].values():  # updates: {node: what the node added to the state}
            for m in (update or {}).get("messages", []):
                added.append(m)
                lines = [f"→ {c['name']}({short(json.dumps(c['args'], ensure_ascii=False))})"
                         for c in getattr(m, "tool_calls", [])]
                if isinstance(m, ToolMessage):
                    lines.append(f"← {m.name} {m.status}: {short(m.text)}")
                if lines:
                    print(("\n" if mid_line else "") + "\n".join(lines), flush=True)
                    mid_line = False
    print()
    return added

## 3. Demo
Run the cell: the tool calls (`→`), their results (`←`) and the answer appear while the agent works; the `INFO` lines
are the tools' log.

The demo is one turn, so the question is a plain string: edit it and run the cell again. Reading the user's input
(`input()`, a chat UI) is only needed when the agent talks back: multi-turn, resume, human-in-the-loop (Session 02).

In [5]:
question = "What was the revenue by product category in Q3 2026 (July to September)?"
messages = await stream(agent, question)

→ list_tables({})


INFO | list_tables() -> ok 0 ms


← list_tables success: ["customers", "order_items", "orders", "products"]
→ describe_table({"table": "order_items"})


INFO | describe_table('order_items') -> ok 0 ms


← describe_table success: {"table": "order_items", "columns": [{"name": "order_id", "type": "INTEGER"}, {"name": "product_id", "type": "…
→ describe_table({"table": "products"})
→ describe_table({"table": "orders"})


INFO | describe_table('orders') -> ok 1 ms
INFO | describe_table('products') -> ok 7 ms


← describe_table success: {"table": "orders", "columns": [{"name": "id", "type": "INTEGER"}, {"name": "customer_id", "type": "INTEGER"},…
← describe_table success: {"table": "products", "columns": [{"name": "id", "type": "INTEGER"}, {"name": "name", "type": "TEXT"}, {"name"…
→ run_query({"sql": "SELECT p.category, SUM(oi.quantity * oi.unit_price) AS revenue_vnd\nFROM orders o\nJOIN order_items o…)


INFO | run_query("SELECT p.category, SUM(oi.quantity * oi.unit_price) AS revenue_vnd\nF…) -> ok 3 ms


← run_query success: {"rows": [{"category": "Books", "revenue_vnd": 83211300}, {"category": "Electronics", "revenue_vnd": 525634500…
Q3 2026 revenue by product category (excluding cancelled orders):

- Books: **83,211,300 VND**
- Electronics: **525,634,500 VND**
- Home: **576,707,000 VND**
- Toys: **137,280,500 VND**


### A wrong column: the agent fixes its own call
The first `run_query` fails, the error lists the valid columns, the model rewrites the query.

In [6]:
question = ("A colleague sent this query for the delivered revenue of September 2026. Run it as is first; "
            "if it fails, fix it and give me the number: "
            "SELECT SUM(total_amount) FROM orders WHERE status = 'delivered' AND created_at LIKE '2026-09%'")
messages = await stream(agent, question)
statuses = [m.status for m in messages if isinstance(m, ToolMessage)]
assert "error" in statuses and statuses[-1] == "success"  # an error came back, then a call succeeded

→ list_tables({})


INFO | list_tables() -> ok 1 ms


← list_tables success: ["customers", "order_items", "orders", "products"]
→ describe_table({"table": "orders"})


INFO | describe_table('orders') -> ok 0 ms


← describe_table success: {"table": "orders", "columns": [{"name": "id", "type": "INTEGER"}, {"name": "customer_id", "type": "INTEGER"},…
→ describe_table({"table": "order_items"})


...

← describe_table success: {"table": "order_items", "columns": [{"name": "order_id", "type": "INTEGER"}, {"name": "product_id", "type": "…
→ run_query({"sql": "SELECT SUM(total_amount) FROM orders WHERE status = 'delivered' AND created_at LIKE '2026-09%'"})


INFO | run_query("SELECT SUM(total_amount) FROM orders WHERE status = 'delivered' AND c…) -> error 0 ms


← run_query error: no such column: total_amount. Valid columns: customers(id, name, email, city, created_at); order_items(order_i…
→ run_query({"sql": "SELECT SUM(oi.quantity * oi.unit_price) AS delivered_revenue_vnd FROM orders o JOIN order_items oi ON…)


INFO | run_query("SELECT SUM(oi.quantity * oi.unit_price) AS delivered_revenue_vnd FROM…) -> ok 0 ms


← run_query success: {"rows": [{"delivered_revenue_vnd": 315880500}], "row_count": 1}
The query failed because `orders` has no `total_amount` column. Using `SUM(quantity × unit_price)` for delivered orders created in September 2026, the revenue is **315,880,500 VND**.


### A request to delete data
The model may refuse, or try `run_query`; either way nothing can be written.

In [7]:
messages = await stream(agent, "Delete all cancelled orders from the database.")
with closing(connect_readonly()) as con:
    print("orders in the database:", con.execute("SELECT count(*) FROM orders").fetchone()[0])  # still 1500

I can’t delete orders; this database access is read-only.
orders in the database: 1500
